# Evaluating Embeddings — Classification & Regression

Tests all three embedders on both tasks. Produces a 2×3 grid of t-SNE plots and prints clustering statistics for all 6 combinations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.manifold import TSNE
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
MODEL_NAMES = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "all-roberta-large-v1",
]

# ── Load datasets ────────────────────────────────────────────────────────────
df_cls = pd.read_csv("dataset/final_binary/final_dataset.csv", low_memory=False)
df_cls = df_cls[["question", "slm_accuracy_fraction"]].dropna().copy()
df_cls["question"] = df_cls["question"].astype(str)
df_cls["label"] = (df_cls["slm_accuracy_fraction"] < 0.75).astype(int)

df_reg = pd.read_csv("dataset/final_regressor/final_qwens_dataset.csv", low_memory=False)
df_reg = df_reg[["question", "slm_accuracy_fraction"]].dropna().copy()
df_reg["question"] = df_reg["question"].astype(str)

# ── Load simple questions (all models answer correctly → accuracy = 1.0) ─────
df_simple = pd.read_csv("dataset/Simple_questions/questions.csv")
df_simple = df_simple[["question"]].dropna().copy()
df_simple["question"] = df_simple["question"].astype(str)
df_simple["slm_accuracy_fraction"] = 1.0
df_simple["label"] = 0  # simple

N_MAIN   = len(df_cls)
N_SIMPLE = len(df_simple)

# Combined frames used for embedding and t-SNE
df_cls = pd.concat([df_cls, df_simple[["question", "slm_accuracy_fraction", "label"]]], ignore_index=True)
df_reg = pd.concat([df_reg, df_simple[["question", "slm_accuracy_fraction"]]], ignore_index=True)

# Boolean mask identifying the simple-questions rows
simple_mask = np.zeros(len(df_cls), dtype=bool)
simple_mask[N_MAIN:] = True

print(f"Classification dataset: {len(df_cls)} rows  ({N_MAIN} main + {N_SIMPLE} simple)")
print(f"Regression dataset:     {len(df_reg)} rows")

Classification dataset: 37429 rows  (36429 main + 1000 simple)
Regression dataset:     37429 rows


In [ ]:
# ── Embed dataset once per model (both CSVs have identical questions) ─────────
# results[model_name] = embeddings_array  (shared for cls & reg label views)
results = {}

for model_name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Encoding with {model_name}")
    model = SentenceTransformer(model_name, device="cuda")

    emb = model.encode(
        df_cls["question"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    print(f"  Embeddings: {emb.shape}")
    results[model_name] = emb

print("\nAll embeddings computed.")


Encoding with all-MiniLM-L6-v2


Batches:   0%|          | 0/1170 [00:00<?, ?it/s]

  Embeddings: (37429, 384)

Encoding with all-mpnet-base-v2


Batches:   0%|          | 0/1170 [00:00<?, ?it/s]

  Embeddings: (37429, 768)

Encoding with all-roberta-large-v1


Batches:   0%|          | 0/1170 [00:00<?, ?it/s]

  Embeddings: (37429, 1024)

All embeddings computed.


In [ ]:
# ── t-SNE reduction — one projection per model ───────────────────────────────
# tsne_2d[model_name] = array(N, 2)  — reused for all label views
tsne_2d = {}

for model_name in MODEL_NAMES:
    print(f"t-SNE: {model_name} …")
    reducer = TSNE(n_components=2, perplexity=30, metric="euclidean", random_state=42)
    tsne_2d[model_name] = reducer.fit_transform(results[model_name])

print("\nt-SNE done.")

t-SNE: all-MiniLM-L6-v2 …
t-SNE: all-mpnet-base-v2 …
t-SNE: all-roberta-large-v1 …

t-SNE done.


In [ ]:
# ── 2×3 grid plot ────────────────────────────────────────────────────────────
# Rows: Classification (top) | Regression (bottom)
# Cols: MiniLM | mpnet | roberta
# Both rows share the same t-SNE projection; only the colouring differs.

label_map = {0: "simple", 1: "complex"}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, model_name in enumerate(MODEL_NAMES):
    xy = tsne_2d[model_name]

    # ── Classification row (row 0) ────────────────────────────────────────
    ax = axes[0, col]
    for lbl, grp_df in df_cls.groupby("label"):
        idx = grp_df.index
        ax.scatter(xy[idx, 0], xy[idx, 1], s=5, alpha=0.6, label=label_map[lbl])
    ax.set_title(f"Classification\n{model_name}", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(title="Label", markerscale=2, fontsize=8)

    # ── Regression row (row 1) ────────────────────────────────────────────
    ax = axes[1, col]
    sc = ax.scatter(
        xy[:, 0], xy[:, 1],
        c=df_reg["slm_accuracy_fraction"].values.astype(float),
        cmap="viridis",
        s=5,
        alpha=0.6,
    )
    plt.colorbar(sc, ax=ax, label="accuracy fraction")
    ax.set_title(f"Regression\n{model_name}", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("t-SNE Projections of Embeddings (Classification vs Regression)", fontsize=14, y=1.01)
plt.tight_layout()
# plt.savefig("evaluating_embeddings.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ── t-SNE colored by dataset name ────────────────────────────────────────────
df_cls_full = pd.read_csv("dataset/final_binary/final_dataset.csv", usecols=["dataset_name"], low_memory=False)
dataset_names = np.concatenate([
    df_cls_full["dataset_name"].values,
    np.full(N_SIMPLE, "Simple_questions"),
])

unique_datasets = sorted(set(dataset_names))
palette = plt.cm.tab10.colors
color_map = {name: palette[i % len(palette)] for i, name in enumerate(unique_datasets)}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for col, model_name in enumerate(MODEL_NAMES):
    ax = axes[col]
    xy = tsne_2d[model_name]
    for name in unique_datasets:
        mask = dataset_names == name
        ax.scatter(xy[mask, 0], xy[mask, 1], s=5, alpha=0.5, label=name, color=color_map[name])
    ax.set_title(model_name, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(title="Dataset", markerscale=3, fontsize=9)

fig.suptitle("t-SNE Projections Colored by Dataset Name", fontsize=14)
plt.tight_layout()
# plt.savefig("evaluating_embeddings_by_dataset.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ── Individual mpnet-base plots ───────────────────────────────────────────────
MPNET = "all-mpnet-base-v2"
xy = tsne_2d[MPNET]

# 1. Classification (simple / complex)
fig, ax = plt.subplots(figsize=(8, 7))
for lbl, grp_df in df_cls.groupby("label"):
    idx = grp_df.index
    ax.scatter(xy[idx, 0], xy[idx, 1], s=5, alpha=0.6, label=label_map[lbl])
ax.set_title("all-mpnet-base-v2 — Classification (simple / complex)", fontsize=12)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(title="Label", markerscale=3, fontsize=10)
plt.tight_layout()
# plt.savefig("mpnet_classification.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. Regression (continuous accuracy fraction)
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=df_reg["slm_accuracy_fraction"].values.astype(float),
    cmap="viridis",
    s=5,
    alpha=0.6,
)
plt.colorbar(sc, ax=ax, label="accuracy fraction")
ax.set_title("all-mpnet-base-v2 — Regression (accuracy fraction)", fontsize=12)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
# plt.savefig("mpnet_regression.png", dpi=200, bbox_inches="tight")
plt.show()

# 3. Colored by dataset name
fig, ax = plt.subplots(figsize=(8, 7))
for name in unique_datasets:
    mask = dataset_names == name
    ax.scatter(xy[mask, 0], xy[mask, 1], s=5, alpha=0.5, label=name, color=color_map[name])
ax.set_title("all-mpnet-base-v2 — Colored by Dataset Name", fontsize=12)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(title="Dataset", markerscale=3, fontsize=9)
plt.tight_layout()
# plt.savefig("mpnet_by_dataset.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ── Statistics for all 6 combinations ────────────────────────────────────────
# Three label sets: full combined, main-only, simple-only
cls_labels     = df_cls["label"].values
reg_labels_raw = df_reg["slm_accuracy_fraction"].values.astype(float)
reg_labels_binned = pd.qcut(
    df_reg.loc[~simple_mask, "slm_accuracy_fraction"], q=4, labels=False, duplicates="drop"
)  # main rows only (used for silhouette in main-only section)

SAMPLE_N = 10000

def pairwise_spearman(emb, labels, n=SAMPLE_N, random_state=42):
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(emb), size=min(n, len(emb)), replace=False)
    emb_s, lab_s = emb[idx], labels[idx]
    emb_dist = cosine_distances(emb_s).ravel()
    lab_dist = np.abs(lab_s[:, None] - lab_s[None, :]).ravel()
    mask = emb_dist > 0
    return spearmanr(emb_dist[mask], lab_dist[mask]).statistic

header = f"{'Model':<25} {'Subset':<14} {'Task':<16} {'Silhouette':>12} {'DB / Spearman':>15}"
sep    = "-" * len(header)
print(header)
print(sep)

for model_name in MODEL_NAMES:
    emb      = results[model_name]
    emb_main = emb[~simple_mask]
    emb_simp = emb[simple_mask]

    # ── Full combined (main + simple) ─────────────────────────────────────
    # Classification: all rows (simple → label 0, complex → label 1)
    sil = silhouette_score(emb, cls_labels, sample_size=5000, random_state=42)
    db  = davies_bouldin_score(emb, cls_labels)
    print(f"{model_name:<25} {'combined':<14} {'Classification':<16} {sil:>12.4f} {db:>15.4f}  (DB ↓)")

    # Regression: all rows, accuracy_fraction
    reg_binned_full = pd.qcut(reg_labels_raw, q=4, labels=False, duplicates="drop")
    sil = silhouette_score(emb, reg_binned_full, sample_size=5000, random_state=42)
    sp  = pairwise_spearman(emb, reg_labels_raw)
    print(f"{model_name:<25} {'combined':<14} {'Regression':<16} {sil:>12.4f} {sp:>15.4f}  (Spearman ↑)")

    # ── Main dataset only ─────────────────────────────────────────────────
    cls_main = cls_labels[~simple_mask]
    sil = silhouette_score(emb_main, cls_main, sample_size=5000, random_state=42)
    db  = davies_bouldin_score(emb_main, cls_main)
    print(f"{model_name:<25} {'main only':<14} {'Classification':<16} {sil:>12.4f} {db:>15.4f}  (DB ↓)")

    reg_main_raw = reg_labels_raw[~simple_mask]
    sil = silhouette_score(emb_main, reg_labels_binned, sample_size=5000, random_state=42)
    sp  = pairwise_spearman(emb_main, reg_main_raw)
    print(f"{model_name:<25} {'main only':<14} {'Regression':<16} {sil:>12.4f} {sp:>15.4f}  (Spearman ↑)")

    # ── Simple questions only (cosine spread as a proxy for cluster compactness) ─
    mean_cosine_dist = cosine_distances(emb_simp).mean()
    print(f"{model_name:<25} {'simple only':<14} {'Spread (cos)':<16} {mean_cosine_dist:>12.4f} {'—':>15}  (↓ = tighter cluster)")

    print(sep)

print()
print("Silhouette:        higher is better  (range −1 … 1)")
print("Davies-Bouldin:    lower is better   (range 0 … ∞)")
print("Pairwise Spearman: higher is better  (range −1 … 1)  — embedding dist vs label dist")
print("Cosine spread:     lower is better   — mean pairwise cosine distance within simple_questions")